In [1]:
!pip install ultralytics

In [1]:
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import albumentations as A
import cv2 as cv

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DATASET_ROOT = Path("/content/drive/MyDrive/InfraDrone/RDD2022/")
YOLO_ROOT = Path("/content/drive/MyDrive/InfraDrone/RDD2022/yolo")

# Hyperparameters
IMG_SIZE = 736
MIN_VISIBILITY = .25
EPOCHS = 100


CLASS_MAP = {
    "longitudinal crack": 0,
    "transverse crack": 1,
    "alligator crack": 2,
    "other corruption": 3,
    "pothole": 4,
}

In [7]:
class ConditionalTransforms(A.ImageOnlyTransform):  
    def __init__(self, drone_transforms, non_drone_transforms, always_apply=False, p=1.0):  
        super().__init__(always_apply, p)  
          
        self.drone_transforms =drone_transforms
          
        self.non_drone_transforms = non_drone_transforms
      
    def __call__(self, force_apply=False, **kwargs):  
        # Access the image path from the labels dictionary  
        if 'im_file' in kwargs:  
            image_path = kwargs['im_file']  
            path = str(image_path or "").lower()  
              
            if "drone" in path:  
                return self.drone_transforms(**kwargs)  
            else:  
                return self.non_drone_transforms(**kwargs)  
          
        # Fallback: return original data if no path found  
        return kwargs

In [8]:
drone_like_transforms = A.Compose(
    [
        # Crop out sky/horizon/hood if present; focus on road surface
        A.RandomResizedCrop(
            size=(640, 640), scale=(0.35, 0.85), ratio=(0.75, 1.35), p=0.8
        ),
        # Mild perspective warp, not insane
        A.Perspective(scale=(0.03, 0.12), keep_size=True, fit_output=False, p=0.5),
        # Simulate drone/camera rotation
        A.ShiftScaleRotate(
            shift_limit=0.08,
            scale_limit=(-0.25, 0.15),  # zoom out more often than in
            rotate_limit=25,
            border_mode=cv.BORDER_CONSTANT,
            p=0.7,
        ),
        # Simulate imperfect drone footage
        A.OneOf(
            [
                A.MotionBlur(blur_limit=5, p=1.0),
                A.GaussianBlur(blur_limit=3, p=1.0),
            ],
            p=0.25,
        ),
        # Lighting differences from outdoor drone footage
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.HueSaturationValue(
            hue_shift_limit=5, sat_shift_limit=20, val_shift_limit=20, p=0.3
        ),
        # Camera noise/compression
        A.OneOf(
            [
                A.GaussNoise(p=1.0),
                A.ImageCompression(p=1.0),
            ],
            p=0.25,
        ),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=MIN_VISIBILITY,
    ),
)

In [9]:
conditional_aug = [ConditionalTransforms(A.Compose([]), drone_like_transforms, p=0.2)]

In [3]:
local_dataset = Path("/Users/lukepitstick/Projects/Data-Science/InfraDrone/python/src/models/data/local_dataset")

In [ ]:
model = YOLO("yolo26s.pt")

model.train(data= local_dataset / "converted.yaml", 
            epochs=EPOCHS, 
            imgsz=IMG_SIZE, 
            batch=16, workers=8, 
            device=0,
            augmentations=drone_like_transforms
            )

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=Compose([
  RandomResizedCrop(p=0.8, area_for_downscale=None, interpolation=1, mask_interpolation=0, ratio=(0.75, 1.35), scale=(0.35, 0.85), size=(640, 640)),
  Perspective(p=0.5, border_mode=0, fill=0.0, fill_mask=0.0, fit_output=False, interpolation=1, keep_size=True, mask_interpolation=0, scale=(0.03, 0.12)),
  ShiftScaleRotate(p=0.7, shift_limit_x=(-0.08, 0.08), shift_limit_y=(-0.08, 0.08), scale_limit=(-0.25, 0.1499999999999999), rotate_limit=(-25.0, 25.0), interpolation=1, border_mode=0, fill=0.0, fill_mask=0.0, rotate_method='largest_box', mask_interpolation=0),
  OneOf([
    MotionBlur(p=1.0, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 5), direction_range=(-1.0, 1.0)),
    GaussianBlur(p=1.0, blur_limit=(0, 3), sigma_limit=(0.5, 3.0)),
  ], p=0.25),
  RandomBrightnessContrast(p=0.5,

KeyboardInterrupt: 